# Qwen2.5 Coder C++ Review QLoRA Training

Run this notebook on Kaggle with GPU enabled. It supports single GPU and multi-GPU training through Accelerate. Re-running the training cell resumes automatically from the latest checkpoint in `outputs/`.

In [ ]:
!nvidia-smi
!python --version
!python -m pip install --upgrade uv

In [ ]:
from pathlib import Path

candidates = [Path('/kaggle/working/fyp8th_clean'), Path('/kaggle/input/fyp8th-clean'), Path.cwd()]
repo = next((p for p in candidates if (p / 'pyproject.toml').exists()), Path.cwd())
%cd {repo}
print('Repository:', repo)

In [ ]:
!uv sync --extra gpu --extra export --extra dev

In [ ]:
!uv run python scripts/merge_datasets.py
!wc -l cleaned/merged_cleaned.jsonl
!uv run --extra dev pytest

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/train_qlora.yaml')
config = yaml.safe_load(config_path.read_text())
config['training']['output_dir'] = '/kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora'
config['data']['data_files'] = ['cleaned/merged_cleaned.jsonl']
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config['training']['output_dir'])

In [ ]:
%%bash
set -euo pipefail
NUM_GPUS=$(uv run python - <<'PY'
import torch
print(torch.cuda.device_count())
PY
)
echo "Detected GPUs: ${NUM_GPUS}"
if [ "${NUM_GPUS}" -gt 1 ]; then
  uv run accelerate launch \
    --config_file configs/accelerate_multi_gpu.yaml \
    --num_processes "${NUM_GPUS}" \
    train.py --config configs/train_qlora.yaml
else
  uv run accelerate launch \
    --config_file configs/accelerate_single_gpu.yaml \
    train.py --config configs/train_qlora.yaml
fi

The training cell is pause/resume safe. If Kaggle disconnects or you stop the run, start the notebook again and rerun the same cell. `train.py` automatically resumes from the newest `checkpoint-*` directory in the configured output folder.

In [ ]:
!ls -lah /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora
!find /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora -maxdepth 2 -type f \( -name '*.pth' -o -name 'adapter_model.safetensors' \) -print

In [ ]:
!uv run python evaluate.py \
  --config configs/train_qlora.yaml \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter

In [ ]:
!uv run python merge_lora.py \
  --base-model Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter \
  --output-dir /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged

In [ ]:
!uv run python export_onnx.py \
  --model /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged \
  --output /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review.onnx